# Public UN Comtrade API Exploration

**Objective:** Explore the structure and analytical meaning of a small response from the public UN Comtrade preview API.

The notebook sends a bounded annual merchandise-trade request, inspects the response envelope and source schema, proposes a provisional business grain, and preserves the raw JSON response.

**Reference:** [UN Comtrade TradeFlow](https://comtradeplus.un.org/TradeFlow)

## 1. Environment & Setup

The opening cell imports the HTTP and DataFrame libraries used in the exploration and records their installed versions.

## 2. Public API Scope

The initial request retrieves Germany's total annual merchandise exports to the World aggregate for 2023.

This is an introductory source exploration rather than a production extraction workflow. A single aggregate response can reveal the API structure and available attributes, but it cannot establish completeness or fully validate the business grain.

### Source & Documentation

The notebook uses the unauthenticated UN Comtrade public preview endpoint. The source interface and broader trade-data context are available through [UN Comtrade TradeFlow](https://comtradeplus.un.org/TradeFlow).

## 3. Request Design

The endpoint uses three required path parameters:

| Parameter | Meaning | Selection |
|---|---|---|
| `typeCode` | Trade type: commodities (`C`) or services (`S`) | `C` |
| `freqCode` | Annual (`A`) or monthly (`M`) frequency | `A` |
| `clCode` | Trade classification, such as `HS`, `SS`, `BE`, or `EB` | `HS` |

The query adds the reporting period and requested trade dimensions:

- `period=2023`
- `reporterCode=276` — Germany
- `partnerCode=0` — World aggregate
- `cmdCode=TOTAL` — all commodities
- `flowCode=X` — exports
- `partner2Code=0`, `customsCode=C00`, and `motCode=0` — aggregate breakdowns
- `maxRecords=500` — public-preview response limit used by this request

Endpoint template:

`https://comtradeapi.un.org/public/v1/preview/{typeCode}/{freqCode}/{clCode}[?reporterCode][&period][&partnerCode][&cmdCode][&flowCode]`

The expected response is one aggregate observation for Germany's total merchandise exports to the World in 2023.

## 4. Execute the Preview Request

Send the unauthenticated request with a 30-second timeout, then inspect the HTTP status, prepared URL, and a bounded response preview.

In [2]:
url = "https://comtradeapi.un.org/public/v1/preview/C/A/HS"
# set up query parameters
params = {
    "period": "2023",
    "reporterCode": "276",
    "cmdCode": "TOTAL",
    "flowCode": "X",
    "partnerCode": "0",
    "partner2Code": "0",
    "customsCode": "C00",
    "motCode": "0",
    "maxRecords": 500,
    "breakdownMode": "classic",
    "includeDesc": "true",
}
# get response
response = requests.get(
    url,
    params=params,
    timeout=30,
)
# check status, URL sent, respose preview. So far looks in order, response resembles a dict-like structure.
print("Status:", response.status_code)
print("Request URL:", response.url)
print("Response preview:")
print(response.text[:500])

Status: 200
Request URL: https://comtradeapi.un.org/public/v1/preview/C/A/HS?period=2023&reporterCode=276&cmdCode=TOTAL&flowCode=X&partnerCode=0&partner2Code=0&customsCode=C00&motCode=0&maxRecords=500&breakdownMode=classic&includeDesc=true
Response preview:
{"elapsedTime":"0.05 secs","count":1,"data":[{"typeCode":"C","freqCode":"A","refPeriodId":20230101,"refYear":2023,"refMonth":52,"period":"2023","reporterCode":276,"reporterISO":"DEU","reporterDesc":"Germany","flowCode":"X","flowDesc":"Export","partnerCode":0,"partnerISO":"W00","partnerDesc":"World","partner2Code":0,"partner2ISO":"W00","partner2Desc":"World","classificationCode":"H6","classificationSearchCode":"HS","isOriginalClassification":true,"cmdCode":"TOTAL","cmdDesc":"All Commodities","agg


## 5. Inspect the Response Envelope

Validate the HTTP response and parse the JSON object. The top-level fields describe processing time, returned-record count, trade observations, and any API error.

In [3]:
# check if response is successful
response.raise_for_status()
# parse data, this should get us a dictionary
payload = response.json()
# check the type and dictionary keys
print('Response type: ', type(payload))
print('Dictionary keys: ', payload.keys())

Response type:  <class 'dict'>
Dictionary keys:  dict_keys(['elapsedTime', 'count', 'data', 'error'])


In [4]:
# elapsedTime - how long API took to process request
# count - how many records are returned
# data - includes actual observations
# error - tells whether something went wrong
# check what type of values the keys include
payload_structure = {key: type(value) for key, value in payload.items()}
payload_structure

{'elapsedTime': str, 'count': int, 'data': list, 'error': str}

In [5]:
print("Error:", payload.get("error"))
print("Count:", payload.get("count"))
print("Elapsed time:", payload.get("elapsedTime"))

Error: 
Count: 1
Elapsed time: 0.05 secs


## 6. Inspect the Record & Source Schema

Extract the observations from the response's `data` list and inspect the returned record. The subsequent cells normalize the same source data into a DataFrame, review its 47 attributes and inferred types, and identify missing values.

In [6]:
# separate the data
records = payload.get("data", [])

In [7]:
if not isinstance(records, list):
    raise TypeError("Expected payload['data'] to be a list.")

print("Number of records:", len(records))

Number of records: 1


In [8]:
# inspect the first data point, in this case also the only one
if records:
    first_record = records[0]
    display(first_record)
else:
    print("No trade records were returned.")

{'typeCode': 'C',
 'freqCode': 'A',
 'refPeriodId': 20230101,
 'refYear': 2023,
 'refMonth': 52,
 'period': '2023',
 'reporterCode': 276,
 'reporterISO': 'DEU',
 'reporterDesc': 'Germany',
 'flowCode': 'X',
 'flowDesc': 'Export',
 'partnerCode': 0,
 'partnerISO': 'W00',
 'partnerDesc': 'World',
 'partner2Code': 0,
 'partner2ISO': 'W00',
 'partner2Desc': 'World',
 'classificationCode': 'H6',
 'classificationSearchCode': 'HS',
 'isOriginalClassification': True,
 'cmdCode': 'TOTAL',
 'cmdDesc': 'All Commodities',
 'aggrLevel': 0,
 'isLeaf': False,
 'customsCode': 'C00',
 'customsDesc': 'TOTAL CPC',
 'mosCode': '0',
 'motCode': 0,
 'motDesc': 'TOTAL MOT',
 'qtyUnitCode': -1,
 'qtyUnitAbbr': 'N/A',
 'qty': 0.0,
 'isQtyEstimated': False,
 'altQtyUnitCode': -1,
 'altQtyUnitAbbr': 'N/A',
 'altQty': 0.0,
 'isAltQtyEstimated': False,
 'netWgt': None,
 'isNetWgtEstimated': False,
 'grossWgt': 0.0,
 'isGrossWgtEstimated': False,
 'cifvalue': None,
 'fobvalue': 1725527288354.528,
 'primaryValue': 1

### Record Interpretation

The record represents Germany's total annual merchandise exports to the World in 2023.

Key dimensions include:

- `typeCode=C` and `freqCode=A`: annual merchandise trade
- `reporterCode=276`: Germany
- `flowCode=X`: exports
- `partnerCode=0`: World aggregate
- `classificationCode=H6`: returned Harmonized System revision
- `cmdCode=TOTAL` and `aggrLevel=0`: all commodities at the highest aggregation level
- `customsCode=C00` and `motCode=0`: aggregate customs and transport breakdowns

For this export record, `fobvalue` equals `primaryValue`, while `cifvalue` is unavailable. The primary trade value is approximately USD 1.726 trillion.

Quantity and weight fields are absent or not analytically meaningful at this aggregate level. A recorded zero should not automatically be interpreted as a measured physical quantity. Reporting, aggregation, and estimation flags are retained as source attributes rather than treated as proof of completeness.

In [9]:
if records:
    print(first_record.keys())

dict_keys(['typeCode', 'freqCode', 'refPeriodId', 'refYear', 'refMonth', 'period', 'reporterCode', 'reporterISO', 'reporterDesc', 'flowCode', 'flowDesc', 'partnerCode', 'partnerISO', 'partnerDesc', 'partner2Code', 'partner2ISO', 'partner2Desc', 'classificationCode', 'classificationSearchCode', 'isOriginalClassification', 'cmdCode', 'cmdDesc', 'aggrLevel', 'isLeaf', 'customsCode', 'customsDesc', 'mosCode', 'motCode', 'motDesc', 'qtyUnitCode', 'qtyUnitAbbr', 'qty', 'isQtyEstimated', 'altQtyUnitCode', 'altQtyUnitAbbr', 'altQty', 'isAltQtyEstimated', 'netWgt', 'isNetWgtEstimated', 'grossWgt', 'isGrossWgtEstimated', 'cifvalue', 'fobvalue', 'primaryValue', 'legacyEstimationFlag', 'isReported', 'isAggregate'])


In [10]:
# the record doesn't have a nested object, but still it is common practive to normalize
records_df = pd.json_normalize(records)
print("Shape:", records_df.shape)
display(records_df.head())

Shape: (1, 47)


,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,20230101,2023,52,2023,276,DEU,Germany,X,...,None,False,0.0,False,None,1.725527e+12,1.725527e+12,0,False,True


In [11]:
records_df.dtypes

typeCode                        str
freqCode                        str
refPeriodId                   int64
refYear                       int64
refMonth                      int64
period                          str
reporterCode                  int64
reporterISO                     str
reporterDesc                    str
flowCode                        str
flowDesc                        str
partnerCode                   int64
partnerISO                      str
partnerDesc                     str
partner2Code                  int64
partner2ISO                     str
partner2Desc                    str
classificationCode              str
classificationSearchCode        str
isOriginalClassification       bool
cmdCode                         str
cmdDesc                         str
aggrLevel                     int64
isLeaf                         bool
customsCode                     str
customsDesc                     str
mosCode                         str
motCode                     

In [12]:
missing_summary = (
    records_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)
missing_summary.head(5)

cifvalue    1
netWgt      1
typeCode    0
refYear     0
refMonth    0
dtype: int64

## 7. Assess the Candidate Grain

The source dimensions suggest a composite key that is broader than reporter, period, and flow alone.

The provisional candidate grain is:

**Type × Frequency × Period × Reporter × Partner × Secondary Partner × Flow × Classification × Commodity × Customs Procedure × Transport Mode**

All candidate columns are present, and the single returned record has no duplicate key. Because the response contains only one highly aggregated observation, this is a structural hypothesis rather than a validated grain for larger or more detailed requests.

In [13]:
candidate_grain = [
    "typeCode",
    "freqCode",
    "period",
    "reporterCode",
    "partnerCode",
    "partner2Code",
    "flowCode",
    "classificationCode",
    "cmdCode",
    "customsCode",
    "motCode",
]

available_grain_columns = [
    column
    for column in candidate_grain
    if column in records_df.columns
]

missing_grain_columns = [
    column
    for column in candidate_grain
    if column not in records_df.columns
]

print("Available candidate columns:", available_grain_columns)
print("Missing candidate columns:", missing_grain_columns)

duplicate_count = records_df.duplicated(
    subset=available_grain_columns
).sum()

print("Candidate duplicates:", duplicate_count)

Available candidate columns: ['typeCode', 'freqCode', 'period', 'reporterCode', 'partnerCode', 'partner2Code', 'flowCode', 'classificationCode', 'cmdCode', 'customsCode', 'motCode']
Missing candidate columns: []
Candidate duplicates: 0


## 8. Preserve the Raw Response

Locate the repository root, create the ignored `data/raw/` directory when needed, and save the complete source response with a timestamped filename. The raw payload is preserved without analytical transformation.

In [14]:
from pathlib import Path
Path.cwd()

WindowsPath('c:/Users/ozgur/GitHub/global-trade-flow-pipeline/notebooks')

In [15]:
# find project root
def find_project_root(start_path: Path = Path.cwd()) -> Path:
    current = start_path.resolve()
    for candidate in [current, *current.parents,]:
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root.")
project_root = find_project_root()
project_root

WindowsPath('C:/Users/ozgur/GitHub/global-trade-flow-pipeline')

In [16]:
import json
from datetime import datetime
# get current timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# create or select the raw data directory
raw_directory = (
    project_root
    / "data"
    / "raw"
)
raw_directory.mkdir(
    parents=True,
    exist_ok=True,
)
# create outputh file path
output_path = raw_directory / (
    f"comtrade_deu_world_exports_"
    f"2023_{timestamp}.json"
)

In [17]:
# save raw response always, we may need to investigate source later and it should be available
with output_path.open("w", encoding="utf-8") as file:
    json.dump(
        payload,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Saved raw response to:")
print(output_path)

Saved raw response to:
C:\Users\ozgur\GitHub\global-trade-flow-pipeline\data\raw\comtrade_deu_world_exports_2023_20260803_213123.json


## 9. Reuse the Public-Preview Client

The remaining cells repeat the exploration through the project's reusable client while keeping authentication disabled.

They demonstrate:

- the same bounded Germany request;
- timestamped raw-response storage;
- a second annual request for Türkiye; and
- three separate monthly requests for Germany's January–March 2023 exports.

The monthly calls remain separate because the public preview interface is deliberately limited. These examples extend the source exploration without claiming complete coverage or production-scale extraction.

In [18]:
# Reusable Extraction

import sys
from pathlib import Path

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / ".git").exists()
)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import ComtradeSettings
from src.extract import (
    ComtradeClient,
    TradeQuery,
    save_raw_payload,
)

# Keep this notebook explicitly on the public preview API.
preview_client = ComtradeClient(ComtradeSettings(api_key=None))

In [19]:
# Test a Germany request
germany_payload = preview_client.fetch_trade_data(TradeQuery(
    type_code="C",
    frequency_code="A",
    classification_code="HS",
    period="2023",
    reporter_code="276",
    partner_code="0",
    commodity_code="TOTAL",
    flow_code="X",
))
# inspect the response
print("Count:", germany_payload["count"])
print("Error:", germany_payload["error"])

Count: 1
Error: 


In [21]:
output_path = save_raw_payload(
    germany_payload,
    output_directory=project_root / "data" / "raw",
    reporter_code="276",
    partner_code="0",
    flow_code="X",
    period="2023",
)

Saved raw response to:
c:\Users\ozgur\GitHub\global-trade-flow-pipeline\data\raw\comtrade_reporter-276_partner-0_flow-X_period-2023_20260803_213200.json


In [22]:
# Test a Turkiye request
turkiye_payload = preview_client.fetch_trade_data(TradeQuery(
    type_code="C",
    frequency_code="A",
    classification_code="HS",
    period="2023",
    reporter_code="792",
    partner_code="0",
    commodity_code="TOTAL",
    flow_code="X",
))
# inspect the response
turkiye_record = turkiye_payload["data"][0]
print(
    "Reporter:",
    turkiye_record["reporterDesc"],
)
print(
    "Primary value:",
    turkiye_record["primaryValue"],
)

Reporter: Türkiye
Primary value: 255627429011.0


In [23]:
# The client handles the preview API's request pacing.
monthly_records = []

for month in ["202301", "202302", "202303"]:
    payload = preview_client.fetch_trade_data(TradeQuery(
        type_code="C",
        frequency_code="M",
        classification_code="HS",
        period=month,
        reporter_code="276",
        partner_code="0",
        commodity_code="TOTAL",
        flow_code="X",
    ))

    monthly_records.extend(payload["data"])

In [24]:
monthly_records[0]

{'typeCode': 'C',
 'freqCode': 'M',
 'refPeriodId': 20230101,
 'refYear': 2023,
 'refMonth': 1,
 'period': '202301',
 'reporterCode': 276,
 'reporterISO': 'DEU',
 'reporterDesc': 'Germany',
 'flowCode': 'X',
 'flowDesc': 'Export',
 'partnerCode': 0,
 'partnerISO': 'W00',
 'partnerDesc': 'World',
 'partner2Code': 0,
 'partner2ISO': 'W00',
 'partner2Desc': 'World',
 'classificationCode': 'H6',
 'classificationSearchCode': 'HS',
 'isOriginalClassification': True,
 'cmdCode': 'TOTAL',
 'cmdDesc': 'All Commodities',
 'aggrLevel': 0,
 'isLeaf': False,
 'customsCode': 'C00',
 'customsDesc': 'TOTAL CPC',
 'mosCode': '0',
 'motCode': 0,
 'motDesc': 'TOTAL MOT',
 'qtyUnitCode': -1,
 'qtyUnitAbbr': 'N/A',
 'qty': 0.0,
 'isQtyEstimated': False,
 'altQtyUnitCode': -1,
 'altQtyUnitAbbr': 'N/A',
 'altQty': 0.0,
 'isAltQtyEstimated': False,
 'netWgt': None,
 'isNetWgtEstimated': False,
 'grossWgt': 0.0,
 'isGrossWgtEstimated': False,
 'cifvalue': None,
 'fobvalue': 139553698569.058,
 'primaryValue': 1

In [25]:
monthly_df = pd.json_normalize(monthly_records)
monthly_df[["period", "reporterDesc", "flowDesc", "primaryValue"]]

,period,reporterDesc,flowDesc,primaryValue
0,202301,Germany,Export,1.395537e+11
1,202302,Germany,Export,1.422445e+11
2,202303,Germany,Export,1.633963e+11


## 10. Conclusion

The public preview endpoint returned a successful aggregate response with one observation and 47 source attributes. The response separated coded dimensions, descriptions, monetary values, physical measures, and reporting flags, while `primaryValue` provided the main analytical trade value.

The candidate grain includes reporter, period, flow, partner, commodity, classification, customs, transport, and related dimensions. Its uniqueness remains provisional because the initial request returned only one aggregate row.

The raw payload was preserved, and the reusable public-preview client successfully handled additional annual and single-month examples. These results establish the API's basic structure and usability, but they do not demonstrate exhaustive coverage, larger-query behavior, or source-data accuracy.